In [0]:
# ============================================================
# CELL 1: Generate Synthetic Healthcare Data
# ============================================================

# We import these libraries:
# - pandas: to create and manipulate our dataset easily
# - numpy: for random number generation
# - random & datetime: to generate realistic fake data
#

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set a seed so results are reproducible (same data every run)
random.seed(42)
np.random.seed(42)

# --- Define realistic options for our columns ---
diagnoses = [
    "Diabetes", "Hypertension", "Asthma", "COVID-19",
    "Heart Failure", "Pneumonia", "Kidney Disease", "Cancer"
]

hospitals = [
    "City General Hospital", "Green Valley Medical",
    "Sunrise Health Center", "Metro Care Hospital", "Lakeside Clinic"
]

genders = ["Male", "Female", "Other"]

# --- Function to generate one patient record ---
def generate_patient(patient_id):
    admission = datetime(2022, 1, 1) + timedelta(days=random.randint(0, 700))
    discharge  = admission + timedelta(days=random.randint(1, 30))
    
    return {
        "patient_id"     : f"P{patient_id:04d}",          # e.g. P0001
        "name"           : f"Patient_{patient_id}",        # anonymized
        "age"            : random.randint(18, 90),
        "gender"         : random.choice(genders),
        "diagnosis"      : random.choice(diagnoses),
        "admission_date" : admission.strftime("%Y-%m-%d"),
        "discharge_date" : discharge.strftime("%Y-%m-%d"),
        "hospital"       : random.choice(hospitals),
        "treatment_cost" : round(random.uniform(500, 50000), 2),
        "readmitted"     : random.choice(["Yes", "No"])
    }

# --- Generate 1000 patient records ---
patients_data = [generate_patient(i) for i in range(1, 1001)]

# Convert list of dicts → pandas DataFrame
pdf = pd.DataFrame(patients_data)

# Preview first 5 rows
print(f"Total Records: {len(pdf)}")
pdf.head()

Total Records: 1000


,patient_id,name,age,gender,diagnosis,admission_date,discharge_date,hospital,treatment_cost,readmitted
0,P0001,Patient_1,21,Other,Heart Failure,2023-10-17,2023-10-21,Green Valley Medical,11548.93,Yes
1,P0002,Patient_2,87,Male,Kidney Disease,2023-11-24,2023-12-18,City General Hospital,1974.96,Yes
2,P0003,Patient_3,21,Other,COVID-19,2022-08-27,2022-09-13,Lakeside Clinic,21266.23,No
3,P0004,Patient_4,18,Male,Kidney Disease,2023-08-27,2023-09-05,Sunrise Health Center,14254.63,Yes
4,P0005,Patient_5,29,Female,Hypertension,2022-12-11,2022-12-15,Sunrise Health Center,42450.97,No


In [0]:
# ============================================================
# CELL 2 (FIXED): Save CSV using dbutils + spark writer
# ============================================================

# In Databricks Community Edition, directly writing to /dbfs/ 
# via Python's os/open can throw I/O errors.
# The correct approach is to convert pandas df → Spark df → write via Spark
# OR use dbutils to work with DBFS safely.

# Step A: Convert pandas DataFrame → Spark DataFrame
df_spark_temp = spark.createDataFrame(pdf)

# Step B: Write it as a CSV to DBFS using Spark's writer
# This is the reliable way to land files in DBFS
df_spark_temp.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("dbfs:/healthcare_project/raw/patients_raw.csv")

print("✅ CSV saved successfully to DBFS using Spark writer!")

# Verify the file exists
display(dbutils.fs.ls("dbfs:/healthcare_project/raw/"))

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-8078678025667757>, line 18
     11 df_spark_temp = spark.createDataFrame(pdf)
     13 # Step B: Write it as a CSV to DBFS using Spark's writer
     14 # This is the reliable way to land files in DBFS
     15 df_spark_temp.write \
     16     .mode("overwrite") \
     17     .option("header", "true") \
---> 18     .csv("dbfs:/healthcare_project/raw/patients_raw.csv")
     20 print("✅ CSV saved successfully to DBFS using Spark writer!")
     22 # Verify the file exists

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:831, in DataFrameWriter.csv(self, path, mode, compression, sep, quote, escape, header, nullValue, escapeQuotes, quoteAll, dateFormat, timestampFormat, ignoreLeadingWhiteSpace, ignoreTrailingWhiteSpace, charToEscapeQuoteEscaping, encoding, emptyValue, lineSep)
    812 self.m

In [0]:
# ============================================================
# CELL 2 (NEW): Create Database/Schema
# ============================================================

# Since DBFS public root is disabled, we skip CSV entirely.
# We'll work directly with managed Delta tables — 
# this is actually the MODERN and PREFERRED approach in Databricks!

# Create a database (also called schema) to hold all our tables
spark.sql("CREATE DATABASE IF NOT EXISTS healthcare_db")
spark.sql("USE healthcare_db")

print("✅ Database 'healthcare_db' ready!")

✅ Database 'healthcare_db' ready!


In [0]:
# ============================================================
# CELL 3 (NEW): Convert pandas → Spark → Save as Bronze Delta Table directly
# ============================================================

# createDataFrame() converts our pandas DataFrame 
# into a distributed Spark DataFrame
df_bronze = spark.createDataFrame(pdf)

# Write directly as a MANAGED Delta table
# Managed = Databricks controls the storage location (no path needed!)
# This completely avoids DBFS restrictions

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_db.bronze_patients")  # ← managed table

print("✅ Bronze Delta table created successfully!")
print(f"✅ Total rows: {df_bronze.count()}")

✅ Bronze Delta table created successfully!
✅ Total rows: 1000


In [0]:
# ============================================================
# CELL 4 (UPDATED): Preview and Validate
# ============================================================

# Now read from the managed Delta table instead of a file path
df_bronze = spark.table("healthcare_db.bronze_patients")

display(df_bronze)

print(f"Total Rows : {df_bronze.count()}")
print(f"Total Cols : {len(df_bronze.columns)}")

from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

print("\n🔍 Null counts per column:")
display(null_counts)

patient_id,name,age,gender,diagnosis,admission_date,discharge_date,hospital,treatment_cost,readmitted
P0001,Patient_1,21,Other,Heart Failure,2023-10-17,2023-10-21,Green Valley Medical,11548.93,Yes
P0002,Patient_2,87,Male,Kidney Disease,2023-11-24,2023-12-18,City General Hospital,1974.96,Yes
P0003,Patient_3,21,Other,COVID-19,2022-08-27,2022-09-13,Lakeside Clinic,21266.23,No
P0004,Patient_4,18,Male,Kidney Disease,2023-08-27,2023-09-05,Sunrise Health Center,14254.63,Yes
P0005,Patient_5,29,Female,Hypertension,2022-12-11,2022-12-15,Sunrise Health Center,42450.97,No
P0006,Patient_6,76,Other,Hypertension,2022-02-14,2022-03-10,Metro Care Hospital,4400.61,No
P0007,Patient_7,64,Other,COVID-19,2023-10-06,2023-10-26,City General Hospital,2768.31,Yes
P0008,Patient_8,47,Male,Kidney Disease,2022-10-24,2022-10-27,Sunrise Health Center,22943.81,No
P0009,Patient_9,63,Male,Heart Failure,2022-06-16,2022-06-28,City General Hospital,30651.98,Yes
P0010,Patient_10,49,Male,Cancer,2023-07-01,2023-07-25,Metro Care Hospital,13862.42,Yes


Total Rows : 1000
Total Cols : 10

🔍 Null counts per column:


patient_id,name,age,gender,diagnosis,admission_date,discharge_date,hospital,treatment_cost,readmitted
0,0,0,0,0,0,0,0,0,0


In [0]:
# ============================================================
# CELL 5 (NEW): Verify with SQL
# ============================================================

# Since it's already a managed Delta table, 
# we can query it with SQL directly — no extra registration needed!

result = spark.sql("""
    SELECT 
        diagnosis,
        COUNT(*) AS total_patients
    FROM healthcare_db.bronze_patients
    GROUP BY diagnosis
    ORDER BY total_patients DESC
""")

display(result)

diagnosis,total_patients
Hypertension,145
Cancer,143
Asthma,124
Kidney Disease,123
Heart Failure,121
COVID-19,117
Pneumonia,115
Diabetes,112
